# 03 - Temporal candidate strategy

## Objetivo

Definir o universo supervisionado temporal para o sistema de recomendacao de proximo carrinho.

Este notebook gera candidatos e targets usando janelas temporais por usuario. Para cada pedido alvo, o historico disponivel termina antes desse pedido.

Cada usuario elegivel gera exatamente tres janelas temporais no conjunto `prior`:

- Janela 1: `split = train`.
- Janela 2: `split = train`.
- Janela 3: `split = validation`.

O pedido do `eval_set = train` original do Instacart e preservado como holdout final:

- Prior completo do usuario: `split = test`.

O dataset final sera usado pelos notebooks seguintes para feature engineering temporal, baselines e treino do modelo de ranking.

Artefato gerado:

- `data/features/temporal_candidates_v1/` - candidatos temporais particionados em Parquet com target.

---

## 1. Setup inicial

In [1]:
import gc
import time
from collections import defaultdict
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

pd.set_option("display.max_columns", None)

In [ ]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TEST_SAMPLE_DIR = DATA_DIR / "test_sample"
FEATURES_DIR = DATA_DIR / "features"
CACHED_DIR = DATA_DIR / "cache"

UNIFIED_DATASET_PATH = PROCESSED_DIR / "orders_product_unified.parquet"
SAMPLE_DATASET_PATH = TEST_SAMPLE_DIR / "orders_product_sample.parquet"

TEMPORAL_CANDIDATES_DIR = FEATURES_DIR / "temporal_candidates_v1"
JACCARD_NEIGHBORS_PATH = CACHED_DIR / "temporal_jaccard_neighbors_v1.parquet"

MIN_PRIOR_ORDERS = 4
TEMPORAL_WINDOWS_PER_USER = 3
TRAIN_WINDOWS_PER_USER = 2
CAP = 200

P50_THRESHOLD = 48
P90_THRESHOLD = 139

GROUP_CONFIG = {
    "P0-P50": {"recompra": 50, "similarity": 150},
    "P50-P90": {"recompra": 125, "similarity": 75},
    "P90+": {"recompra": 160, "similarity": 40},
}

GLOBAL_POOL_SIZE = 200
CATEGORY_TOP_AISLES = 10
CATEGORY_AISLE_DEPTH = 15 # máx 150 candidatos de categoria por usuário
N_NEIGHBORS = 30
MAX_CANDIDATE_NEIGHBORS = 100
MIN_SHARED_PRODUCTS = 5
JACCARD_MATRIX_CHUNK_SIZE = 500
SIMILARITY_POOL_SIZE = 500
CHUNK_USER_COUNT = 2_500
PROGRESS_EVERY_CHUNKS = 5
PROGRESS_EVERY_USERS = 10_000
OVERWRITE_TEMPORAL_CANDIDATES = False
OVERWRITE_JACCARD_CACHE = False

assert UNIFIED_DATASET_PATH.exists(), (
    f"Dataset unificado nao encontrado em: {UNIFIED_DATASET_PATH}. "
    "Execute o notebook 01 antes de continuar."
)

print("Arquivos de entrada encontrados com sucesso.")
print(f"Output temporal sera salvo em: {TEMPORAL_CANDIDATES_DIR}")

Arquivos de entrada encontrados com sucesso.
Output temporal sera salvo em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/temporal_candidates_v1


### 1.1 Carregamento dos dados

Sao carregadas apenas as colunas necessarias para construcao das janelas, candidatos e target.

In [3]:
USE_SAMPLE = False

selected_columns = [
    "order_id",
    "user_id",
    "eval_set",
    "order_number",
    "product_id",
    "aisle_id",
    "department_id",
]

dataset_path = SAMPLE_DATASET_PATH if USE_SAMPLE else UNIFIED_DATASET_PATH

df = pd.read_parquet(dataset_path, columns=selected_columns)

print(f"Dataset carregado: {dataset_path.name}")
print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")

if USE_SAMPLE:
    print(
        "\nATENCAO: amostra auxiliar carregada. "
        "Nao use este resultado para conclusoes finais."
    )

Dataset carregado: orders_product_unified.parquet
Shape: 33,819,106 linhas x 7 colunas


In [4]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 7 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   order_id       int64 
 1   user_id        int64 
 2   eval_set       object
 3   order_number   int64 
 4   product_id     int64 
 5   aisle_id       int64 
 6   department_id  int64 
dtypes: int64(6), object(1)
memory usage: 3.2 GB


In [5]:
df.head()

,order_id,user_id,eval_set,order_number,product_id,aisle_id,department_id
0,2,202279,prior,3,33120,86,16
1,2,202279,prior,3,28985,83,4
2,2,202279,prior,3,9327,104,13
3,2,202279,prior,3,45918,19,13
4,2,202279,prior,3,30035,17,13


---

## 2. Definicao das janelas temporais

### 2.1 Separacao prior / test holdout

O conjunto `prior` sera usado para gerar janelas temporais de treino e validacao.

O conjunto `eval_set = train` original sera tratado como holdout e mantido como split `test` final.

In [6]:
df_prior = df[df["eval_set"] == "prior"].copy()
df_test = df[df["eval_set"] == "train"].copy()

assert not df_prior.empty, "Nenhum registro prior encontrado."
assert not df_test.empty, "Nenhum registro de test holdout encontrado."

print(f"Prior: {df_prior.shape[0]:,} linhas")
print(f"Test holdout: {df_test.shape[0]:,} linhas")

Prior: 32,434,489 linhas
Test holdout: 1,384,617 linhas


### 2.2 Usuarios elegiveis

Um usuario elegivel precisa ter:

- pelo menos 4 pedidos no `prior`;
- exatamente 1 pedido no holdout `eval_set = train`.

Com 4 pedidos `prior`, e possivel gerar exatamente 3 janelas temporais: os ultimos 3 pedidos `prior` viram targets temporais.

In [7]:
prior_orders = (
    df_prior[["user_id", "order_id", "order_number"]]
    .drop_duplicates()
    .sort_values(["user_id", "order_number"])
    .reset_index(drop=True)
)

test_orders = (
    df_test[["user_id", "order_id", "order_number"]]
    .drop_duplicates()
    .sort_values(["user_id", "order_number"])
    .reset_index(drop=True)
)

prior_orders_per_user = prior_orders.groupby("user_id")["order_id"].size()
test_orders_per_user = test_orders.groupby("user_id")["order_id"].size()

users_with_enough_prior = set(
    prior_orders_per_user[prior_orders_per_user >= MIN_PRIOR_ORDERS].index
)
users_with_single_test = set(
    test_orders_per_user[test_orders_per_user == 1].index
)

eligible_users = sorted(users_with_enough_prior & users_with_single_test)

assert eligible_users, "Nenhum usuario elegivel encontrado."

eligibility_summary = pd.DataFrame(
    [
        {
            "users_with_prior": prior_orders_per_user.shape[0],
            "users_with_enough_prior": len(users_with_enough_prior),
            "users_with_single_test": len(users_with_single_test),
            "eligible_users": len(eligible_users),
        }
    ]
)

eligibility_summary

,users_with_prior,users_with_enough_prior,users_with_single_test,eligible_users
0,206209,182223,131209,115909


### 2.3 Criacao das janelas prior

Para cada usuario elegivel, os ultimos 3 pedidos `prior` sao selecionados como targets temporais.

As duas primeiras janelas entram em `train`; a ultima entra em `validation`.

Exemplo:

| Quantidade de pedidos `prior` | Janelas geradas |
|---:|---|
| 4 | pedido 2 -> train, pedido 3 -> train, pedido 4 -> validation |
| 8 | pedido 6 -> train, pedido 7 -> train, pedido 8 -> validation |

Em todos os casos, o historico usado para gerar candidatos termina no pedido anterior ao pedido alvo da janela.

In [8]:
prior_windows = []
eligible_user_set = set(eligible_users)

for user_id, user_orders in prior_orders[
    prior_orders["user_id"].isin(eligible_user_set)
].groupby("user_id", sort=False):
    selected_targets = user_orders.tail(TEMPORAL_WINDOWS_PER_USER).reset_index(drop=True)

    assert len(selected_targets) == TEMPORAL_WINDOWS_PER_USER, (
        f"Usuario {user_id} nao possui {TEMPORAL_WINDOWS_PER_USER} targets prior."
    )

    for window_idx, target_row in selected_targets.iterrows():
        split = "train" if window_idx < TRAIN_WINDOWS_PER_USER else "validation"
        target_order_number = int(target_row["order_number"])
        history_orders = user_orders[
            user_orders["order_number"] < target_order_number
        ]

        prior_windows.append(
            {
                "user_id": int(user_id),
                "split": split,
                "window_number": int(window_idx + 1),
                "target_order_id": int(target_row["order_id"]),
                "target_order_number": target_order_number,
                "history_start_order_number": int(history_orders["order_number"].min()),
                "history_end_order_number": int(history_orders["order_number"].max()),
                "history_order_count": int(len(history_orders)),
            }
        )

prior_windows_df = pd.DataFrame(prior_windows)

assert len(prior_windows_df) == len(eligible_users) * TEMPORAL_WINDOWS_PER_USER, (
    "Quantidade de janelas prior diferente do esperado."
)
assert prior_windows_df["history_order_count"].ge(1).all(), (
    "Toda janela precisa ter pelo menos um pedido historico."
)
assert (
    prior_windows_df["history_end_order_number"]
    < prior_windows_df["target_order_number"]
).all(), "Ha janela com historico posterior ou igual ao target."

prior_windows_df.groupby("split").size().reset_index(name="windows")

,split,windows
0,train,231818
1,validation,115909


### 2.4 Criacao das janelas de teste

O split `test` usa o pedido do holdout `eval_set = train` como target.

O historico disponivel para teste e todo o conjunto `prior` do usuario.

In [9]:
last_prior_order = (
    prior_orders[prior_orders["user_id"].isin(eligible_user_set)]
    .groupby("user_id", as_index=False)
    .agg(
        history_start_order_number=("order_number", "min"),
        history_end_order_number=("order_number", "max"),
        history_order_count=("order_id", "size"),
    )
)

test_windows_df = test_orders[
    test_orders["user_id"].isin(eligible_user_set)
].merge(
    last_prior_order,
    on="user_id",
    how="inner",
)

test_windows_df = test_windows_df.rename(
    columns={
        "order_id": "target_order_id",
        "order_number": "target_order_number",
    }
)

test_windows_df["split"] = "test"
test_windows_df["window_number"] = TEMPORAL_WINDOWS_PER_USER + 1

test_windows_df = test_windows_df[
    [
        "user_id",
        "split",
        "window_number",
        "target_order_id",
        "target_order_number",
        "history_start_order_number",
        "history_end_order_number",
        "history_order_count",
    ]
]

assert len(test_windows_df) == len(eligible_users), (
    "Quantidade de janelas de teste diferente do numero de usuarios elegiveis."
)
assert (
    test_windows_df["history_end_order_number"]
    < test_windows_df["target_order_number"]
).all(), "Ha janela test com historico posterior ou igual ao target."

test_windows_df.head()

,user_id,split,window_number,target_order_id,target_order_number,history_start_order_number,history_end_order_number,history_order_count
0,1,test,4,1187899,11,1,10,10
1,2,test,4,1492625,15,1,14,14
2,5,test,4,2196797,5,1,4,4
3,7,test,4,525192,21,1,20,20
4,10,test,4,1822501,6,1,5,5


In [10]:
temporal_windows_df = pd.concat(
    [prior_windows_df, test_windows_df],
    ignore_index=True,
)

temporal_windows_df["user_window_id"] = (
    temporal_windows_df["user_id"].astype(str)
    + "_"
    + temporal_windows_df["split"].astype(str)
    + "_"
    + temporal_windows_df["target_order_id"].astype(str)
)

assert temporal_windows_df["user_window_id"].is_unique, (
    "user_window_id duplicado."
)

window_summary = (
    temporal_windows_df
    .groupby("split")
    .agg(
        windows=("user_window_id", "count"),
        users=("user_id", "nunique"),
        min_history_orders=("history_order_count", "min"),
        median_history_orders=("history_order_count", "median"),
        max_history_orders=("history_order_count", "max"),
    )
    .reset_index()
)

window_summary

,split,windows,users,min_history_orders,median_history_orders,max_history_orders
0,test,115909,115909,4,11.0,99
1,train,231818,115909,1,8.0,97
2,validation,115909,115909,3,10.0,98


---

## 3. Artefatos auxiliares para geracao de candidatos

### 3.1 Base de referencia para popularidade, categoria e similaridade

A base de referencia (`split = train`) usa apenas historicos das janelas de treino.

Ela evita usar o target de validacao e o target de teste para construir pools globais de fallback.

Neste notebook, popularidade global e popularidade por categoria nao sao recalculadas para cada janela. Elas sao calculadas uma vez a partir da base de referencia. Essa e uma aproximacao pragmatica: o dataset possui uma ordem temporal clara por usuario, mas nao uma linha do tempo global simples para dizer quais produtos eram populares no mercado no momento exato de cada compra.

Essa decisao reduz custo computacional e evita usar diretamente os targets de validacao/teste na construcao dos pools auxiliares.

In [11]:
train_history_end = (
    temporal_windows_df[temporal_windows_df["split"] == "train"]
    .groupby("user_id", as_index=False)["history_end_order_number"]
    .max()
)

reference_df = df_prior.merge(
    train_history_end,
    on="user_id",
    how="inner",
    suffixes=("", "_max"),
)

reference_df = reference_df[
    reference_df["order_number"] <= reference_df["history_end_order_number"]
].copy()

assert not reference_df.empty, "Base de referencia vazia."

print(f"Base de referencia: {len(reference_df):,} linhas")
print(f"Usuarios na base de referencia: {reference_df['user_id'].nunique():,}")
print(f"Produtos na base de referencia: {reference_df['product_id'].nunique():,}")

Base de referencia: 17,794,365 linhas
Usuarios na base de referencia: 115,909
Produtos na base de referencia: 49,160


### 3.2 Popularidade global de referencia

In [12]:
global_popularity = (
    reference_df
    .groupby("product_id")
    .size()
    .reset_index(name="reference_purchase_count")
    .sort_values(
        ["reference_purchase_count", "product_id"],
        ascending=[False, True],
    )
    .head(GLOBAL_POOL_SIZE)
    .reset_index(drop=True)
)

global_pool_list = global_popularity["product_id"].astype(int).tolist()

assert global_pool_list, "Pool global vazio."

print(f"Pool global: {len(global_pool_list):,} produtos")
global_popularity.head()

Pool global: 200 produtos


,product_id,reference_purchase_count
0,24852,261839
1,13176,208241
2,21137,148233
3,21903,133655
4,47209,121076


### 3.3 Popularidade por categoria de referencia

In [13]:
aisle_product_popularity = (
    reference_df
    .groupby(["aisle_id", "product_id"])
    .size()
    .reset_index(name="reference_purchase_count")
    .sort_values(
        ["aisle_id", "reference_purchase_count", "product_id"],
        ascending=[True, False, True],
    )
)

aisle_product_pool = {
    int(aisle_id): group["product_id"].astype(int).head(CATEGORY_AISLE_DEPTH).tolist()
    for aisle_id, group in aisle_product_popularity.groupby("aisle_id", sort=False)
}

assert aisle_product_pool, "Pool por aisle vazio."

print(f"Aisles com pool de categoria: {len(aisle_product_pool):,}")
aisle_product_popularity.head()

Aisles com pool de categoria: 134


,aisle_id,product_id,reference_purchase_count
68,1,26047,3392
64,1,25199,2735
56,1,22281,2373
52,1,21560,2082
61,1,23719,1447


### 3.4 Estruturas para similaridade Jaccard

**Por que a Similaridade de Jaccard?**

Trabalhamos com **feedback implícito** (dados binários: comprou [1] ou não comprou [0]). O Jaccard é a escolha ideal para este cenário por três motivos:

* **Ignora Co-ausências (Zeros):** Se dois usuários não compraram os mesmos 40.000 produtos do site, isso não significa que eles são parecidos. Diferente da *Distância Euclidiana*, o Jaccard ignora os zeros compartilhados.
* **Focado em Dados Binários:** Métodos como a *Correlação de Pearson* dependem de notas (ex: 1 a 5 estrelas) e perdem o sentido matemático em matrizes binárias.
* **Penaliza Desproporção:** Ao contrário da *Similaridade de Cosseno*, o Jaccard penaliza severamente a comparação entre um usuário comum (comprou 3 itens) e um "super-comprador/revendedor" (comprou 1.000 itens), evitando recomendações distorcidas.

A similaridade entre o conjunto de produtos do Usuário A ($S_A$) e do Usuário B ($S_B$) é calculada por:

$$J(A,B) = \frac{|S_A \cap S_B|}{|S_A \cup S_B|}$$

Onde:
* **$|S_A \cap S_B|$ (Interseção):** Quantidade de produtos comprados por ambos.
* **$|S_A \cup S_B|$ (União):** Total de produtos únicos somando os dois carrinhos ($|S_A| + |S_B| - \text{Interseção}$).

Vantagens neste projeto:

- simples e interpretavel;
- adequado para conjuntos esparsos de produtos;
- gera candidatos personalizados antes do modelo neural;
- aproxima a estrategia de um retrieval colaborativo sem exigir embeddings pre-treinados.

Para reduzir custo na geracao das janelas, o notebook calcula uma matriz esparsa usuario-produto a partir da base de referencia. Em seguida, usa multiplicacao matricial em chunks para obter intersecoes usuario-usuario e salvar um cache de top vizinhos por usuario, sem materializar a matriz usuario-usuario completa em memoria.

Importante: o cache Jaccard e calculado com uma fotografia fixa da base de referencia (`split = train`) e reutilizado nas janelas temporais. Isso introduz um leakage pequeno/controlado para algumas janelas mais antigas, pois os vizinhos podem refletir informacoes agregadas posteriores ao ponto historico daquela janela.

Esse trade-off foi aceito porque:

- o target da propria janela nao entra diretamente na recompra, nas features da janela ou no calculo do target;
- recalcular vizinhos Jaccard para cada janela teria custo muito alto;
- a abordagem se aproxima melhor de producao, onde vizinhos e pools auxiliares seriam calculados em batch e consultados online;
- o cache reduz latencia e evita reconstruir a similaridade durante a geracao de candidatos.

**Workflow**:

1. Criar matriz esparsa usuario-produto com a base de referencia.
2. Calcular intersecoes usuario-usuario em chunks via matriz_chunk * matriz transposta.
3. Filtrar vizinhos com pelo menos MIN_SHARED_PRODUCTS = 5 produtos em comum.
4. Manter no maximo MAX_CANDIDATE_NEIGHBORS = 100 possiveis vizinhos por usuario.
5. Calcular Jaccard e salvar os N_NEIGHBORS = 30 vizinhos mais similares.
6. Na geracao de candidatos, usar os produtos comprados por esses vizinhos, excluindo produtos ja presentes no historico da janela.


Esse cache se aproxima melhor de uma rotina batch de producao: o custo pesado fica offline e a geracao de candidatos consulta vizinhos pre-computados. O processamento em chunks reduz pico de memoria e evita queda de kernel ao calcular similaridade para muitos usuarios.

In [14]:
reference_pairs = (
    reference_df[["user_id", "product_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

reference_user_products = (
    reference_pairs
    .groupby("user_id")["product_id"]
    .agg(lambda values: set(map(int, values)))
    .to_dict()
)

reference_user_ids = np.array(sorted(reference_user_products))
reference_product_ids = np.array(sorted(reference_pairs["product_id"].unique()))

reference_user_to_idx = {
    int(user_id): idx
    for idx, user_id in enumerate(reference_user_ids)
}
reference_product_to_idx = {
    int(product_id): idx
    for idx, product_id in enumerate(reference_product_ids)
}

row_idx = reference_pairs["user_id"].map(reference_user_to_idx).to_numpy()
col_idx = reference_pairs["product_id"].map(reference_product_to_idx).to_numpy()

user_item_matrix = csr_matrix(
    (
        np.ones(len(reference_pairs), dtype=np.int32),
        (row_idx, col_idx),
    ),
    shape=(len(reference_user_ids), len(reference_product_ids)),
    dtype=np.int32,
)

reference_user_product_counts = np.asarray(
    user_item_matrix.sum(axis=1)
).ravel().astype(np.int32)

print(f"Usuarios com produtos na base de referencia: {len(reference_user_ids):,}")
print(f"Produtos na matriz usuario-produto: {len(reference_product_ids):,}")
print(f"Interacoes unicas na matriz: {user_item_matrix.nnz:,}")

Usuarios com produtos na base de referencia: 115,909
Produtos na matriz usuario-produto: 49,160
Interacoes unicas na matriz: 7,180,693


### 3.5 Cache matricial de vizinhos Jaccard por usuario

O cache evita recalcular similaridade durante a geracao das janelas temporais.

Ele salva os top vizinhos de cada usuario elegivel com base na matriz esparsa usuario-produto da base de referencia. A multiplicacao usuario-usuario e feita em chunks para controlar memoria. Durante a geracao de candidatos, a janela usa esses vizinhos e remove produtos ja comprados naquele historico especifico.

In [15]:
def build_jaccard_neighbors_from_intersection_chunk(
    intersection_chunk,
    user_offset,
):
    jaccard_records = []

    for local_user_idx in range(intersection_chunk.shape[0]):
        user_idx = user_offset + local_user_idx
        row_start = intersection_chunk.indptr[local_user_idx]
        row_end = intersection_chunk.indptr[local_user_idx + 1]

        neighbor_indices = intersection_chunk.indices[row_start:row_end]
        shared_counts = intersection_chunk.data[row_start:row_end].astype(np.int32)

        valid_mask = (
            (neighbor_indices != user_idx)
            & (shared_counts >= MIN_SHARED_PRODUCTS)
        )
        neighbor_indices = neighbor_indices[valid_mask]
        shared_counts = shared_counts[valid_mask]

        if len(neighbor_indices) == 0:
            continue

        candidate_order = np.lexsort(
            (
                reference_user_ids[neighbor_indices],
                -shared_counts,
            )
        )[:MAX_CANDIDATE_NEIGHBORS]

        neighbor_indices = neighbor_indices[candidate_order]
        shared_counts = shared_counts[candidate_order]

        union_counts = (
            reference_user_product_counts[user_idx]
            + reference_user_product_counts[neighbor_indices]
            - shared_counts
        )
        jaccard_scores = shared_counts / union_counts

        score_order = np.lexsort(
            (
                reference_user_ids[neighbor_indices],
                -jaccard_scores,
            )
        )[:N_NEIGHBORS]

        user_id = int(reference_user_ids[user_idx])

        for idx in score_order:
            jaccard_records.append(
                (
                    user_id,
                    int(reference_user_ids[neighbor_indices[idx]]),
                    int(shared_counts[idx]),
                    float(jaccard_scores[idx]),
                )
            )

    return pd.DataFrame(
        jaccard_records,
        columns=[
            "user_id",
            "neighbor_user_id",
            "shared_products",
            "jaccard_score",
        ],
    )

In [16]:
CACHED_DIR.mkdir(parents=True, exist_ok=True)

if JACCARD_NEIGHBORS_PATH.exists() and not OVERWRITE_JACCARD_CACHE:
    jaccard_neighbors_df = pd.read_parquet(JACCARD_NEIGHBORS_PATH)
    print(f"Cache Jaccard carregado: {JACCARD_NEIGHBORS_PATH}")
else:
    jaccard_chunk_dfs = []
    total_reference_users = user_item_matrix.shape[0]

    for chunk_start in range(
        0,
        total_reference_users,
        JACCARD_MATRIX_CHUNK_SIZE,
    ):
        chunk_end = min(
            chunk_start + JACCARD_MATRIX_CHUNK_SIZE,
            total_reference_users,
        )

        intersection_chunk = (
            user_item_matrix[chunk_start:chunk_end]
            @ user_item_matrix.T
        ).tocsr()

        chunk_neighbors_df = build_jaccard_neighbors_from_intersection_chunk(
            intersection_chunk=intersection_chunk,
            user_offset=chunk_start,
        )

        if not chunk_neighbors_df.empty:
            jaccard_chunk_dfs.append(chunk_neighbors_df)

        del intersection_chunk
        del chunk_neighbors_df
        gc.collect()

        processed_users = chunk_end
        if (
            processed_users % PROGRESS_EVERY_USERS == 0
            or processed_users == total_reference_users
        ):
            print(
                f"Usuarios processados no cache Jaccard: "
                f"{processed_users:,}/{total_reference_users:,}"
            )

    if jaccard_chunk_dfs:
        jaccard_neighbors_df = pd.concat(
            jaccard_chunk_dfs,
            ignore_index=True,
        )
    else:
        jaccard_neighbors_df = pd.DataFrame(
            columns=[
                "user_id",
                "neighbor_user_id",
                "shared_products",
                "jaccard_score",
            ]
        )

    jaccard_neighbors_df.to_parquet(JACCARD_NEIGHBORS_PATH, index=False)
    print(f"Cache Jaccard salvo em: {JACCARD_NEIGHBORS_PATH}")

    del jaccard_chunk_dfs
    gc.collect()

print(f"Vizinhos Jaccard: {len(jaccard_neighbors_df):,} linhas")
jaccard_neighbors_df.head()

Cache Jaccard carregado: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/cache/temporal_jaccard_neighbors_v1.parquet
Vizinhos Jaccard: 3,092,236 linhas


,user_id,neighbor_user_id,shared_products,jaccard_score
0,1,10027,8,0.222222
1,1,81456,8,0.222222
2,1,1344,6,0.206897
3,1,24312,7,0.200000
4,1,109507,7,0.200000


In [17]:
jaccard_neighbors_map = {
    int(user_id): list(
        zip(
            group["neighbor_user_id"].astype(int),
            group["jaccard_score"].astype(float),
        )
    )
    for user_id, group in jaccard_neighbors_df.groupby("user_id", sort=False)
}

print(f"Usuarios com vizinhos Jaccard em cache: {len(jaccard_neighbors_map):,}")

Usuarios com vizinhos Jaccard em cache: 108,918


---

## 4. Funcoes auxiliares de candidatos

### 4.1 Segmentacao e alocacao dos candidatos

Cada janela e segmentada pelo numero de produtos unicos no historico disponivel daquela janela.

A segmentacao controla a alocacao inicial entre recompra e similaridade, mantendo `CAP = 200` candidatos por janela.

| Segmento | Regra | Recompra | Similaridade | Total planejado | Overflow |
|---|---:|---:|---:|---:|---|
| P0-P50 | `history_unique_products <= 48` | 50 | 150 | 200 | recompra -> similarity -> category -> global |
| P50-P90 | `49 <= history_unique_products <= 139` | 125 | 75 | 200 | recompra -> similarity -> category -> global |
| P90+ | `history_unique_products >= 140` | 160 | 40 | 200 | recompra -> similarity -> category -> global |

Quando uma fonte nao preenche sua cota, o saldo passa para a proxima fonte da cadeia. Na pratica, usuarios com menos historico tendem a depender mais de similaridade, categoria e popularidade global; usuarios com mais historico tendem a concentrar mais candidatos de recompra.

O preenchimento segue a ordem:

`recompra -> similarity -> category -> global`


Quando uma fonte nao preenche sua cota, o saldo passa para a proxima fonte. Assim, usuarios com pouco historico tendem a cair mais em categoria e popularidade global, enquanto usuarios com historico maior tendem a receber mais candidatos de recompra.

In [18]:
def assign_history_group(unique_product_count):
    if unique_product_count <= P50_THRESHOLD:
        return "P0-P50"
    if unique_product_count <= P90_THRESHOLD:
        return "P50-P90"
    return "P90+"


def fill_from_pool(pool, selected_set, n_needed, source):
    added = []

    for product_id in pool:
        if n_needed <= 0:
            break
        product_id = int(product_id)
        if product_id in selected_set:
            continue

        added.append((product_id, source))
        selected_set.add(product_id)
        n_needed -= 1

    return added

In [19]:
def build_repurchase_pool(history_df):
    repurchase_pool = (
        history_df
        .groupby("product_id", as_index=False)
        .agg(
            purchase_count=("order_id", "size"),
            last_order_number=("order_number", "max"),
        )
        .sort_values(
            ["purchase_count", "last_order_number", "product_id"],
            ascending=[False, False, True],
        )
    )

    return repurchase_pool["product_id"].astype(int).tolist()


def build_category_pool(history_df):
    top_aisles = (
        history_df
        .groupby("aisle_id")
        .size()
        .sort_values(ascending=False)
        .head(CATEGORY_TOP_AISLES)
        .index
        .astype(int)
        .tolist()
    )

    category_pool = []

    for aisle_id in top_aisles:
        category_pool.extend(aisle_product_pool.get(aisle_id, []))

    return category_pool

In [20]:
def build_user_similarity_product_pool(current_user_id):
    neighbor_scores = jaccard_neighbors_map.get(int(current_user_id), [])

    if not neighbor_scores:
        return []

    product_scores = defaultdict(float)

    for neighbor_user_id, jaccard_score in neighbor_scores:
        for product_id in reference_user_products.get(int(neighbor_user_id), set()):
            product_scores[int(product_id)] += float(jaccard_score)

    similarity_pool = [
        product_id
        for product_id, _ in sorted(
            product_scores.items(),
            key=lambda item: (-item[1], item[0]),
        )
    ]

    return similarity_pool[:SIMILARITY_POOL_SIZE]


def build_similarity_pool(history_products, current_user_id, similarity_pool_map):
    user_similarity_pool = similarity_pool_map.get(int(current_user_id), [])

    return [
        product_id
        for product_id in user_similarity_pool
        if product_id not in history_products
    ]

In [21]:
def build_window_candidates(window_row, user_history_df, similarity_pool_map):
    user_id = int(window_row["user_id"])
    target_order_number = int(window_row["target_order_number"])

    history_df = user_history_df[
        user_history_df["order_number"] < target_order_number
    ].copy()

    assert not history_df.empty, (
        f"Historico vazio para user_id={user_id}, target={target_order_number}."
    )

    history_products = set(map(int, history_df["product_id"].unique()))
    history_group = assign_history_group(len(history_products))
    config = GROUP_CONFIG[history_group]

    selected_set = set()
    selected = []

    repurchase_pool = build_repurchase_pool(history_df)
    repurchase_added = fill_from_pool(
        pool=repurchase_pool,
        selected_set=selected_set,
        n_needed=config["recompra"],
        source="recompra",
    )
    selected.extend(repurchase_added)

    shortfall = config["recompra"] - len(repurchase_added)

    similarity_pool = build_similarity_pool(
        history_products=history_products,
        current_user_id=user_id,
        similarity_pool_map=similarity_pool_map,
    )
    similarity_added = fill_from_pool(
        pool=similarity_pool,
        selected_set=selected_set,
        n_needed=config["similarity"] + shortfall,
        source="similarity",
    )
    selected.extend(similarity_added)

    shortfall = CAP - len(selected)

    if shortfall > 0:
        category_pool = build_category_pool(history_df)
        category_added = fill_from_pool(
            pool=category_pool,
            selected_set=selected_set,
            n_needed=shortfall,
            source="category",
        )
        selected.extend(category_added)

    shortfall = CAP - len(selected)

    if shortfall > 0:
        global_added = fill_from_pool(
            pool=global_pool_list,
            selected_set=selected_set,
            n_needed=shortfall,
            source="global",
        )
        selected.extend(global_added)

    records = []

    for candidate_rank, (product_id, candidate_source) in enumerate(
        selected[:CAP],
        start=1,
    ):
        records.append(
            {
                "user_id": user_id,
                "product_id": int(product_id),
                "split": window_row["split"],
                "window_number": int(window_row["window_number"]),
                "user_window_id": window_row["user_window_id"],
                "target_order_id": int(window_row["target_order_id"]),
                "target_order_number": int(window_row["target_order_number"]),
                "history_start_order_number": int(
                    window_row["history_start_order_number"]
                ),
                "history_end_order_number": int(
                    window_row["history_end_order_number"]
                ),
                "history_order_count": int(window_row["history_order_count"]),
                "history_unique_products": int(len(history_products)),
                "history_group": history_group,
                "candidate_source": candidate_source,
                "candidate_rank": int(candidate_rank),
            }
        )

    return records

---

## 5. Geracao particionada dos candidatos temporais

### 5.1 Preparacao do diretorio de saida

In [22]:
if TEMPORAL_CANDIDATES_DIR.exists():
    existing_parts = list(TEMPORAL_CANDIDATES_DIR.glob("*.parquet"))

    if existing_parts:
        if OVERWRITE_TEMPORAL_CANDIDATES:
            print(
                f"Limpando {len(existing_parts):,} particoes antigas em "
                f"{TEMPORAL_CANDIDATES_DIR}"
            )
            for part_file in existing_parts:
                part_file.unlink()
        else:
            raise AssertionError(
                "Diretorio temporal ja contem arquivos e "
                f"OVERWRITE_TEMPORAL_CANDIDATES e False: {TEMPORAL_CANDIDATES_DIR}"
            )
else:
    TEMPORAL_CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Diretorio de saida preparado: {TEMPORAL_CANDIDATES_DIR}")

Diretorio de saida preparado: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/temporal_candidates_v1


### 5.2 Indice dos produtos alvo

In [23]:
target_products_df = (
    df[["order_id", "product_id"]]
    .drop_duplicates()
    .rename(columns={"order_id": "target_order_id"})
)

assert not target_products_df.empty, "Indice de target vazio."

target_products_df.head()

,target_order_id,product_id
0,2,33120
1,2,28985
2,2,9327
3,2,45918
4,2,30035


### 5.3 Geracao por chunks de usuarios

A geracao por chunk foi separada em funcoes pequenas:

- gerar candidatos do chunk;
- aplicar target;
- salvar a particao;
- calcular estatisticas da particao.

O loop final apenas orquestra essas etapas e registra progresso.

Para diagnosticar gargalos, cada particao registra tempo de geracao, target, escrita em Parquet e estatisticas. O pool de produtos vindos dos vizinhos Jaccard e calculado uma vez por usuario dentro do chunk e reutilizado nas janelas daquele usuario.

Pelos tempos observados, o gargalo principal e a geracao dos candidatos, nao a aplicacao do target nem a escrita em Parquet. Isso e esperado porque cada chunk precisa montar candidatos para multiplas janelas por usuario, aplicar deduplicacao, consultar recompra, similaridade, categoria e fallback global.

Em producao, esse custo tende a ser menor por requisicao individual, pois o cache Jaccard e os pools auxiliares ja estariam pre-computados em batch. O processamento pesado deste notebook existe porque estamos materializando o dataset supervisionado completo para treino, validacao e teste offline.

In [26]:
eligible_user_array = np.array(eligible_users)
n_chunks = ceil(len(eligible_user_array) / CHUNK_USER_COUNT)
user_chunks = np.array_split(eligible_user_array, n_chunks)

print(f"Usuarios elegiveis: {len(eligible_user_array):,}")
print(f"Chunks planejados: {len(user_chunks):,}")
print(f"Usuarios por chunk: aproximadamente {CHUNK_USER_COUNT:,}")

Usuarios elegiveis: 115,909
Chunks planejados: 47
Usuarios por chunk: aproximadamente 2,500


In [27]:
def generate_candidates_for_chunk(user_chunk):
    chunk_users = set(map(int, user_chunk.tolist()))

    chunk_windows = temporal_windows_df[
        temporal_windows_df["user_id"].isin(chunk_users)
    ].copy()
    chunk_prior = df_prior[df_prior["user_id"].isin(chunk_users)].copy()

    user_history_map = {
        int(user_id): user_df.sort_values("order_number").copy()
        for user_id, user_df in chunk_prior.groupby("user_id", sort=False)
    }

    similarity_pool_map = {
        user_id: build_user_similarity_product_pool(user_id)
        for user_id in chunk_users
    }

    candidate_records = []

    for window_row in chunk_windows.itertuples(index=False):
        window_dict = window_row._asdict()
        user_history_df = user_history_map[int(window_dict["user_id"])]
        candidate_records.extend(
            build_window_candidates(
                window_row=window_dict,
                user_history_df=user_history_df,
                similarity_pool_map=similarity_pool_map,
            )
        )

    candidates_chunk = pd.DataFrame(candidate_records)

    del chunk_windows
    del chunk_prior
    del user_history_map
    del similarity_pool_map
    del candidate_records

    return candidates_chunk

In [28]:
def add_target_to_candidates(candidates_chunk):
    target_chunk = target_products_df[
        target_products_df["target_order_id"].isin(
            candidates_chunk["target_order_id"].unique()
        )
    ].copy()
    target_chunk["target"] = 1

    candidates_chunk = candidates_chunk.merge(
        target_chunk,
        on=["target_order_id", "product_id"],
        how="left",
    )
    candidates_chunk["target"] = candidates_chunk["target"].fillna(0).astype(int)

    del target_chunk

    return candidates_chunk

In [29]:
def save_candidates_partition(candidates_chunk, chunk_idx):
    part_path = TEMPORAL_CANDIDATES_DIR / f"part-{chunk_idx:04d}.parquet"
    candidates_chunk.to_parquet(part_path, index=False)

    return part_path


def build_partition_stats(candidates_chunk, part_path):
    candidates_per_window = candidates_chunk.groupby("user_window_id")["product_id"].size()

    return {
        "partition": part_path.name,
        "rows": len(candidates_chunk),
        "users": candidates_chunk["user_id"].nunique(),
        "windows": candidates_chunk["user_window_id"].nunique(),
        "positives": int(candidates_chunk["target"].sum()),
        "min_candidates_per_window": int(candidates_per_window.min()),
        "max_candidates_per_window": int(candidates_per_window.max()),
    }


def process_candidate_chunk(chunk_idx, user_chunk):
    chunk_start_time = time.perf_counter()

    candidates_chunk = generate_candidates_for_chunk(user_chunk)
    generated_at = time.perf_counter()

    assert not candidates_chunk.empty, (
        f"Chunk {chunk_idx} nao gerou candidatos."
    )
    assert not candidates_chunk.duplicated(
        subset=["user_window_id", "product_id"]
    ).any(), f"Candidatos duplicados no chunk {chunk_idx}."

    candidates_chunk = add_target_to_candidates(candidates_chunk)
    targeted_at = time.perf_counter()

    part_path = save_candidates_partition(candidates_chunk, chunk_idx)
    saved_at = time.perf_counter()

    stats = build_partition_stats(candidates_chunk, part_path)
    stats_at = time.perf_counter()

    stats.update(
        {
            "generate_seconds": generated_at - chunk_start_time,
            "target_seconds": targeted_at - generated_at,
            "write_seconds": saved_at - targeted_at,
            "stats_seconds": stats_at - saved_at,
            "total_seconds": stats_at - chunk_start_time,
        }
    )

    del candidates_chunk
    gc.collect()

    return stats

In [ ]:
partition_stats = []

for chunk_idx, user_chunk in enumerate(user_chunks):
    chunk_stats = process_candidate_chunk(
        chunk_idx=chunk_idx,
        user_chunk=user_chunk,
    )
    partition_stats.append(chunk_stats)

    processed_chunks = chunk_idx + 1
    if (
        processed_chunks % PROGRESS_EVERY_CHUNKS == 0
        or processed_chunks == len(user_chunks)
    ):
        print(
            f"Chunks processados: {processed_chunks:,}/{len(user_chunks):,} "
            f"- ultima particao: {chunk_stats['partition']} "
            f"- linhas: {chunk_stats['rows']:,} "
            f"- gerar: {chunk_stats['generate_seconds']:.1f}s "
            f"- target: {chunk_stats['target_seconds']:.1f}s "
            f"- parquet: {chunk_stats['write_seconds']:.1f}s"
        )

partition_stats_df = pd.DataFrame(partition_stats)

partition_stats_df.head()

Chunks processados: 10/47 - ultima particao: part-0009.parquet - linhas: 1,972,800 - gerar: 26.1s - target: 0.7s - parquet: 0.6s
Chunks processados: 20/47 - ultima particao: part-0019.parquet - linhas: 1,972,800 - gerar: 27.4s - target: 0.7s - parquet: 0.6s
Chunks processados: 30/47 - ultima particao: part-0029.parquet - linhas: 1,972,800 - gerar: 26.3s - target: 0.7s - parquet: 0.6s
Chunks processados: 40/47 - ultima particao: part-0039.parquet - linhas: 1,972,800 - gerar: 26.4s - target: 0.7s - parquet: 0.6s
Chunks processados: 47/47 - ultima particao: part-0046.parquet - linhas: 1,972,800 - gerar: 25.9s - target: 0.7s - parquet: 0.6s


,partition,rows,users,windows,positives,min_candidates_per_window,max_candidates_per_window,generate_seconds,target_seconds,write_seconds,stats_seconds,total_seconds
0,part-0000.parquet,1973600,2467,9868,72809,200,200,25.576044,0.656799,0.566959,0.173643,26.973446
1,part-0001.parquet,1973600,2467,9868,71756,200,200,25.302064,0.654695,0.599211,0.173834,26.729804
2,part-0002.parquet,1973600,2467,9868,74116,200,200,25.525919,0.776994,0.550379,0.201050,27.054342
3,part-0003.parquet,1973600,2467,9868,74255,200,200,24.831975,0.753340,0.563384,0.186072,26.334771
4,part-0004.parquet,1973600,2467,9868,73195,200,200,24.721893,0.672093,0.525355,0.180947,26.100287
5,part-0005.parquet,1973600,2467,9868,75936,200,200,24.632873,0.681543,0.526951,0.179475,26.020842
6,part-0006.parquet,1973600,2467,9868,73887,200,200,25.032019,0.705819,0.534036,0.180332,26.452205
7,part-0007.parquet,1972800,2466,9864,74275,200,200,25.256549,0.695809,0.546577,0.179953,26.678888
8,part-0008.parquet,1972800,2466,9864,71113,200,200,27.260278,0.753295,0.538518,0.180772,28.732864
9,part-0009.parquet,1972800,2466,9864,73300,200,200,26.129360,0.735366,0.562655,0.183079,27.610460


---

## 6. Validacoes do dataset temporal de candidatos

### 6.1 Validacao das particoes persistidas

In [31]:
temporal_part_paths = sorted(TEMPORAL_CANDIDATES_DIR.glob("*.parquet"))

assert temporal_part_paths, (
    f"Nenhuma particao temporal encontrada em: {TEMPORAL_CANDIDATES_DIR}"
)

persisted_stats = []
duplicated_pairs_total = 0

for part_path in temporal_part_paths:
    part_df = pd.read_parquet(
        part_path,
        columns=[
            "user_id",
            "product_id",
            "split",
            "user_window_id",
            "target_order_id",
            "target_order_number",
            "history_end_order_number",
            "candidate_source",
            "target",
        ],
    )

    duplicated_pairs_total += int(
        part_df.duplicated(subset=["user_window_id", "product_id"]).sum()
    )

    assert (
        part_df["history_end_order_number"] < part_df["target_order_number"]
    ).all(), f"Leakage temporal detectado em {part_path.name}."

    persisted_stats.append(
        {
            "partition": part_path.name,
            "rows": len(part_df),
            "users": part_df["user_id"].nunique(),
            "windows": part_df["user_window_id"].nunique(),
            "positives": int(part_df["target"].sum()),
        }
    )

    del part_df

assert duplicated_pairs_total == 0, (
    "Existem pares user_window_id-product_id duplicados."
)

persisted_stats_df = pd.DataFrame(persisted_stats)

persisted_stats_df.describe(include="all")

,partition,rows,users,windows,positives
count,47,4.700000e+01,47.000000,47.000000,47.000000
unique,47,NaN,NaN,NaN,NaN
top,part-0000.parquet,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,1.972919e+06,2466.148936,9864.595745,73507.319149
std,NaN,2.878997e+02,0.359875,1.439498,1144.852204
min,NaN,1.972800e+06,2466.000000,9864.000000,71113.000000
25%,NaN,1.972800e+06,2466.000000,9864.000000,72720.500000
50%,NaN,1.972800e+06,2466.000000,9864.000000,73532.000000
75%,NaN,1.972800e+06,2466.000000,9864.000000,74262.500000


### 6.2 Distribuicao por split e fonte de candidatos

In [32]:
split_source_stats = []

for part_path in temporal_part_paths:
    part_df = pd.read_parquet(
        part_path,
        columns=["split", "candidate_source", "target"],
    )

    stats = (
        part_df
        .groupby(["split", "candidate_source"], as_index=False)
        .agg(
            rows=("target", "size"),
            positives=("target", "sum"),
        )
    )
    split_source_stats.append(stats)

    del part_df

split_source_stats_df = (
    pd.concat(split_source_stats, ignore_index=True)
    .groupby(["split", "candidate_source"], as_index=False)
    .sum()
)

split_source_stats_df["positive_rate"] = (
    split_source_stats_df["positives"] / split_source_stats_df["rows"]
)

split_source_stats_df.sort_values(["split", "rows"], ascending=[True, False])

,split,candidate_source,rows,positives,positive_rate
3,test,similarity,14186688,100436,0.007080
2,test,recompra,7597470,763048,0.100434
0,test,category,754885,1694,0.002244
1,test,global,642757,1314,0.002044
7,train,similarity,30508739,341456,0.011192
6,train,recompra,12969974,1394746,0.107537
5,train,global,1879266,3050,0.001623
4,train,category,1005621,1675,0.001666
11,validation,similarity,14598184,104514,0.007159
10,validation,recompra,7167016,739886,0.103235


### 6.3 Recall ceiling dos candidatos temporais

O recall ceiling mede a parcela de produtos do pedido alvo que esta presente no conjunto de candidatos.

In [33]:
target_orders_for_windows = temporal_windows_df[
    ["split", "user_window_id", "target_order_id"]
].copy()

window_target_counts = target_orders_for_windows.merge(
    target_products_df,
    on="target_order_id",
    how="left",
)

window_target_counts = (
    window_target_counts
    .groupby(["split", "user_window_id"], as_index=False)
    .agg(target_products=("product_id", "nunique"))
)

generated_positive_counts = []

for part_path in temporal_part_paths:
    part_df = pd.read_parquet(
        part_path,
        columns=["split", "user_window_id", "target"],
    )

    positives = (
        part_df[part_df["target"] == 1]
        .groupby(["split", "user_window_id"], as_index=False)
        .agg(covered_products=("target", "sum"))
    )
    generated_positive_counts.append(positives)

    del part_df

covered_by_window = (
    pd.concat(generated_positive_counts, ignore_index=True)
    .groupby(["split", "user_window_id"], as_index=False)
    .sum()
)

recall_ceiling_df = window_target_counts.merge(
    covered_by_window,
    on=["split", "user_window_id"],
    how="left",
)
recall_ceiling_df["covered_products"] = (
    recall_ceiling_df["covered_products"].fillna(0).astype(int)
)
recall_ceiling_df["recall_ceiling"] = (
    recall_ceiling_df["covered_products"]
    / recall_ceiling_df["target_products"]
)

recall_ceiling_summary = (
    recall_ceiling_df
    .groupby("split", as_index=False)
    .agg(
        windows=("user_window_id", "count"),
        target_products=("target_products", "sum"),
        covered_products=("covered_products", "sum"),
        mean_recall_ceiling=("recall_ceiling", "mean"),
    )
)
recall_ceiling_summary["weighted_recall_ceiling"] = (
    recall_ceiling_summary["covered_products"]
    / recall_ceiling_summary["target_products"]
)

recall_ceiling_summary

,split,windows,target_products,covered_products,mean_recall_ceiling,weighted_recall_ceiling
0,test,115909,1234735,866492,0.710518,0.701764
1,train,231818,2388250,1740927,0.741014,0.728955
2,validation,115909,1212171,847425,0.708520,0.699097


Este valor nao mede a qualidade do modelo de ranking. Ele mede apenas o limite superior imposto pela geracao de candidatos: se um produto do pedido alvo nao aparece entre os candidatos, nenhum modelo posterior conseguira recomenda-lo.

Por isso, o recall ceiling deve ser analisado antes do MLP. Valores proximos entre `validation` e `test` indicam que a estrategia de candidatos esta relativamente estavel entre os splits.

---

## 7. Persistencia final

In [34]:
print("Candidatos temporais salvos com sucesso.")
print(f"Diretorio: {TEMPORAL_CANDIDATES_DIR}")
print(f"Particoes: {len(temporal_part_paths):,}")
print(f"Usuarios elegiveis: {len(eligible_users):,}")
print(f"Janelas temporais: {temporal_windows_df['user_window_id'].nunique():,}")

Candidatos temporais salvos com sucesso.
Diretorio: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/temporal_candidates_v1
Particoes: 47
Usuarios elegiveis: 115,909
Janelas temporais: 463,636
